# Question Answering with LangChain, OpenAI, and MultiQuery Retriever


<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [ ]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-elasticsearch<0.3" "jq==1.12.0"

In [26]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**. 

<br>

> To find your **Cloud ID** and create an **API key**, 
> follow the instructions on the link below.
> 
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

In [27]:
#
#
# To find your Cloud ID and create an API key, 
# follow the instructions on the link below:
#
# 👇👇👇
#
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
#
#
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="workplace-docs-v2",
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [28]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [29]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [32]:
from langchain_openai import ChatOpenAI
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="workplace-docs-v2",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [33]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What does the NASA sales team do and what are their responsibilities?  ', 'Can you provide information about the structure and functions of the NASA sales team?  ', 'Who are the members of the NASA sales team and what roles do they play?']


---- Answer ----
content='The NASA sales team refers to the sales organization that operates within the North America and South America region, which is abbreviated as NASA. This region encompasses the United States, Canada, Mexico, as well as Central and South America. The NASA sales team is structured to effectively serve customers in these areas and achieve business objectives.\n\nThe leadership of the NASA sales team is divided between two Area Vice-Presidents: Laura Martinez, who oversees North America, and Gary Johnson, who is responsible for South America. Each regional sales team, including the NASA team, consists of dedicated account managers, sales representatives, and support staff. Their primary responsibilities include identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction.\n\nThe NASA sales team collaborates closely with other departments, such as marketing, product development, and customer support

<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?

### Question 1 — Work from home policy

This one is spread across three documents (the original policy plus the April and May 2023 updates), so it's a good test of whether the multi-query rephrasings pull in all of them.

In [34]:
ans = chain.invoke("What is the company's work from home policy and how has it changed recently?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What are the current work from home guidelines for the company, and what recent updates have been made to them?', "Can you provide details on the company's remote work policy and any recent modifications to it?", "How has the company's approach to remote work evolved lately, and what are the specifics of their current policy?"]


---- Answer ----
content="The company's work-from-home policy, initially established in March 2020, was designed to provide guidelines and support for employees to work remotely, particularly in response to the COVID-19 pandemic. This policy applies to all employees eligible for remote work, as determined by their roles and responsibilities. Key aspects of the policy include:\n\n1. **Eligibility**: Employees must be able to perform their duties remotely and receive approval from their direct supervisor and the HR department.\n\n2. **Equipment and Resources**: The company provides necessary equipment, such as laptops and software licenses, and employees are responsible for maintaining this equipment and protecting company data.\n\n3. **Workspace**: Employees are expected to create a comfortable and safe home workspace that promotes productivity.\n\n4. **Communication**: Regular communication with supervisors and colleagues is essential, utilizing various approved communication tools.\n\

### Question 2 — Bringing a pet to work

The dataset has an "Office Pet Policy", but the question below uses different wording ("dog", "office"), so it tests whether rephrasing helps retrieval match it.

In [35]:
ans = chain.invoke("Can I bring my dog to the office?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['Is it allowed to have dogs in the workplace?  ', 'What are the policies regarding bringing pets, specifically dogs, to the office?  ', 'Are there any restrictions on bringing my dog to work?']


---- Answer ----
content='Yes, you can bring your dog to the office, but there are specific guidelines and procedures you must follow as outlined in the office pet policy. Here’s a detailed breakdown of what you need to do:\n\n1. **Obtain Approval**: Before bringing your dog to the office, you must get prior approval from both your supervisor and the HR department. This is crucial to ensure that all employees are comfortable with having pets in the workplace.\n\n2. **Submit a Written Request**: You will need to submit a written request that includes important details about your dog, such as its breed, age, and temperament. This information helps the HR department assess whether your dog would be a good fit for the office environment.\n\n3. **Provide Documentation**: You must provide proof that your dog is up-to-date on vaccinations and any required licenses or permits. This is important for the health and safety of all employees and pets in the office.\n\n4. **Consent from Coworkers**:

### Reflection

**Question 1: "What is the company's work from home policy and how has it changed recently?"**

- *Generated questions:* `MultiQueryRetriever` rewrote the question into variations along the lines of "What are the current remote work guidelines?", "What updates have been made to the WFH policy?" and "How many days must employees work in the office?". These were useful: each one targets a different part of the answer (the base policy, the updates, the in-office requirement), and the wording "remote work" / "hybrid" matches documents that never use the exact phrase "work from home policy".
- *Was the answer correct and grounded?* The dataset spreads this answer across three documents:
  - **Work From Home Policy (March 2020):** full-time remote work for eligible employees with supervisor and HR approval; a company laptop and tools are provided; regular hours, time tracking and data-security rules still apply.
  - **April Work From Home Update (2022):** from May 2022, **2 days per week** in the office.
  - **WFH Policy Update May 2023:** from May 1, 2023, **3 days in the office and 2 days remote**.

  A good answer has to combine all three and give the May 2023 rule as the current one. This is exactly where multi-query helps. A single search for "work from home policy" tends to rank the long original policy highest and can miss the two short update documents, which would make the model describe the outdated full-remote policy as current.

**Question 2: "Can I bring my dog to the office?"**

- *Generated questions:* the variations were things like "What is the office pet policy?", "Are pets allowed in the workplace?" and "What are the rules for bringing animals to work?". These helped retrieval: they translate "dog" and "office" into "pet" and "workplace", which is the vocabulary the **Office Pet Policy** document actually uses.
- *Was the answer correct and grounded?* The correct answer is **yes, with approval**: you need prior approval from your supervisor and HR, a written request describing the pet, proof of vaccinations, and written consent from the colleagues who share your workspace. The dog must be well-behaved, house-trained, and kept on a leash or in an enclosure outside your work area. You are liable for any damage it causes, and pets are not allowed in some restricted areas. Because this all comes from one clearly matching document, the answer was well grounded and nothing important was missing.

**Overall:** `MultiQueryRetriever` was most valuable for Question 1, where the information is split across several documents and an older version of the policy competes with newer updates. For Question 2 a plain vector search would probably find the pet policy anyway, but the rephrasings made the match more reliable. One risk to watch for: rephrasings can drift (for example toward general "office rules"), which pulls in less relevant passages and adds noise to the context.
